# Data Processing & Cleaning

This notebook:
1. Loads the raw reviews dataset and profiles its quality (shape, missing values, duplicates, class balance).
2. Cleans the review text (URL/HTML/emoji removal, lowercasing, punctuation/number removal, stopword removal, lemmatization).
3. Removes duplicate reviews.
4. Saves the cleaned dataset for the training notebook.


In [1]:
import re
import string
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

## 1. Load & Profile the Dataset

In [2]:
# Raw dataset columns: category, rating, label ('CG'=fake/'OR'=genuine), text_
df = pd.read_csv('../data/fake_reviews_dataset.csv')

print("Shape:", df.shape)
print("\nMissing values:\n", df.isna().sum())
print("\nFull duplicate rows:", df.duplicated().sum())
print("Duplicate review texts:", df.duplicated(subset=['text_']).sum())
print("\nLabel distribution:\n", df['label'].value_counts())
df.head()


Shape: (40432, 4)

Missing values:
 category    0
rating      0
label       0
text_       0
dtype: int64

Full duplicate rows: 12
Duplicate review texts: 20

Label distribution:
 label
CG    20216
OR    20216
Name: count, dtype: int64


,category,rating,label,text_
0,Home_and_Kitchen_5,5.0,CG,"Love this! Well made, sturdy, and very comfor..."
1,Home_and_Kitchen_5,5.0,CG,"love it, a great upgrade from the original. I..."
2,Home_and_Kitchen_5,5.0,CG,This pillow saved my back. I love the look and...
3,Home_and_Kitchen_5,1.0,CG,"Missing information on how to use it, but it i..."
4,Home_and_Kitchen_5,5.0,CG,Very nice set. Good quality. We have had the s...


**Findings:** 40,432 rows, no missing values, perfectly balanced classes (50/50 CG vs OR), and only a small number of duplicate reviews (~20) to remove.

## 2. Prepare Columns

In [3]:
df = df[['text_', 'label']].copy()
df.rename(columns={'text_': 'review_text'}, inplace=True)
df.dropna(subset=['review_text'], inplace=True)

before = len(df)
df.drop_duplicates(subset=['review_text'], inplace=True)
print(f"Removed {before - len(df)} duplicate reviews. Remaining: {len(df)}")


Removed 20 duplicate reviews. Remaining: 40412


## 3. Text Cleaning

Each step below removes noise that doesn't help the model distinguish fake vs. genuine reviews, or normalizes text so different forms of the same word are treated the same way:

- **URL removal** — links carry no linguistic signal about authenticity.
- **HTML tag removal** — stray markup from copy-pasted text.
- **Emoji removal** — TF-IDF works on words; emojis add noise for this model.
- **Lowercasing** — treats `Great` and `great` as the same token.
- **Number removal** — raw digits rarely carry generalizable signal here.
- **Punctuation removal** — keeps the vocabulary focused on words.
- **Whitespace normalization** — collapses repeated spaces from the steps above.
- **Stopword removal** — removes very common words (`the`, `is`, `and`, ...) that carry little discriminative meaning.
- **Lemmatization** — reduces words to their base form (`loved`, `loving` → `love`) so the vectorizer doesn't split the same concept into multiple features.


In [4]:
STOPWORDS = set(stopwords.words('english'))
LEMMATIZER = WordNetLemmatizer()

URL_RE = re.compile(r'https?://\S+|www\.\S+')
HTML_RE = re.compile(r'<.*?>')
EMOJI_RE = re.compile(
    "["
    "\U0001F600-\U0001F64F"
    "\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF"
    "\U0001F1E0-\U0001F1FF"
    "\U00002700-\U000027BF"
    "\U0001F900-\U0001F9FF"
    "\U00002600-\U000026FF"
    "]+", flags=re.UNICODE)

def clean_text(text: str) -> str:
    """Clean a single review: strip noise, normalize, lemmatize."""
    text = str(text)
    text = URL_RE.sub(' ', text)
    text = HTML_RE.sub(' ', text)
    text = EMOJI_RE.sub(' ', text)
    text = text.lower()
    text = re.sub(r'\d+', ' ', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()

    words = text.split()
    words = [w for w in words if w.isalpha() and w not in STOPWORDS]
    words = [LEMMATIZER.lemmatize(w) for w in words]
    return ' '.join(words)

df['cleaned_text'] = df['review_text'].apply(clean_text)
df = df[df['cleaned_text'].str.strip() != '']
df.head()


,review_text,label,cleaned_text
0,"Love this! Well made, sturdy, and very comfor...",CG,love well made sturdy comfortable love itvery ...
1,"love it, a great upgrade from the original. I...",CG,love great upgrade original ive mine couple year
2,This pillow saved my back. I love the look and...,CG,pillow saved back love look feel pillow
3,"Missing information on how to use it, but it i...",CG,missing information use great product price
4,Very nice set. Good quality. We have had the s...,CG,nice set good quality set two month


## 4. Encode Labels & Save

In [5]:
# 'OR' (original / genuine) -> 0, 'CG' (computer-generated / fake) -> 1
df['label_num'] = df['label'].apply(lambda x: 1 if x == 'CG' else 0)

df.to_csv('../data/cleaned_fake_reviews.csv', index=False)
print("Saved cleaned data:", df.shape)


Saved cleaned data: (40411, 4)
